# Preliminary modeling using TPM data

## Data loading


Configure root.

In [ ]:
import sys, subprocess
import numpy as np
import pandas as pd
import torch 
import torch.nn as nn
import torch.nn.functional as F
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/singlecell-autoencoder.git"
repo_dir = Path("singlecell-autoencoder")
if COLAB:
    root = Path("/content/singlecell-autoencoder")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

# Use GPU if available
torch.manual_seed(111)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Configure bulk data path.

In [ ]:
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
    fcnts_path = str(data_dir / "fcnts_timezero")

else:
    fcnts_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/fcnts_timezero"

Load TPM data.

In [ ]:
from src.bulk_data import get_all_tpm_data

# Load data and remove NA genes
bulk_df = get_all_tpm_data(fcnts_path = fcnts_path)

## Train-test split

Try training on all data except for the diagonal, then predict the diagonal?

In [ ]:
# Diagonal mask
mask = (bulk_df["drug1_dose"]) == (bulk_df["drug2_dose"])

# Apply mask
X_train = bulk_df[~mask]
X_test = bulk_df[mask]

# Separate metadata and data
meta = bulk_df.iloc[:, ~bulk_df.columns.str.contains("SP")]
X_train = X_train.iloc[:, X_train.columns.str.contains("SP")]
X_test = X_test.iloc[:, X_test.columns.str.contains("SP")]

# Convert to tensors
X_train = torch.tensor(X_train.to_numpy())
X_test = torch.tensor(X_test.to_numpy())

# Data shape
print(f"# of train samples : {X_train.shape[0]}")
print(f"# of test samples  : {X_test.shape[0]}")

## Training

Train to identify optimal epoch number.

In [ ]:
from src.model import SimpleAE
from src.train import train_simple_ae

# Model params
num_genes = X_train.shape[1]
input_dim = num_genes
h_dim = 64
latent_dim = 4

# Train params
epochs = 300
lr = 0.001
batch_size = 8

# Simple AE parameters
model_params = {
    "input_dim": input_dim,
    "h_dim": h_dim,
    "latent_dim": latent_dim
}

model, train_losses, val_losses = train_simple_ae(
    data = X_train,
    batch_size = batch_size,
    epochs = epochs,
    lr = lr,
    model_params = model_params,
    device = device
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses, label = "Train")
plt.plot(val_losses, label = "Validation")
plt.xlabel("Epochs")
plt.ylabel("MSE loss")
plt.legend()

In [ ]:
from sklearn.metrics import r2_score

with torch.no_grad():
    preds = model(X_test.float())

score = r2_score(X_test.numpy().ravel(), preds.numpy().ravel())
plt.scatter(X_test.numpy().ravel(), preds.numpy().ravel())
plt.text(80000, 20000, f"R^2 = {score:.3f}")
plt.title("Autoencoder predictions on dose-matched combination datapoints")
plt.xlabel("True TPM values")
plt.ylabel("Predicted TPM values")